# Text Encoder

The image encoder converts an image into a compact 64-dimensional vector.

We now need the corresponding component for language:

$$
z_T = f_T(T)
$$

where $T$ represents a tokenized caption and:

$$
z_T \in \mathbb{R}^{64}
$$

The goal is for the image encoder and text encoder to produce vectors in the **same shared embedding space**.

The overall pipeline is:

```text
Image
  ↓
Image Encoder
  ↓
Image Embedding ∈ R^64


Caption
  ↓
Tokenizer
  ↓
Token IDs
  ↓
Text Encoder
  ↓
Text Embedding ∈ R^64



# 1. Why Do We Need a Text Encoder?

Images and text have very different raw representations.

An image is a tensor of pixel values:

$$
I \in \mathbb{R}^{32 \times 32 \times 3}
$$

A caption is a sequence of discrete tokens.

For example:

```text
"red circle top-left"
[red, circle, top-left]


# 2. Text Encoder Architecture

Our text encoder will be deliberately small.

The pipeline is:

```text
Token IDs
    ↓
Token Embeddings
    ↓
Add Positional Embeddings
    ↓
Multi-Head Self-Attention
    ↓
Layer Normalization
    ↓
Text Representation
    ↓
Linear Projection
    ↓
L2 Normalization
    ↓
Final Text Embedding

### For A batch
Tokens
[B, L]

      ↓

Token Embedding
[B, L, D]

      ↓

Positional Embedding
[B, L, D]

      ↓

Self-Attention
[B, L, D]

      ↓

Sequence Representation
[B, D]

      ↓

Projection
[B, P]

      ↓

Normalization
[B, P]

where:

$B$ = batch size,
$L$ = sequence length,
$D$ = transformer embedding dimension,
$P$ = shared projection dimension

In [52]:
# Imports required for the text encoder

import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
import sys
from torch.utils.data import DataLoader

In [53]:
# ============================================================
# Text Encoder — Input Setup
# ============================================================




# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


# ------------------------------------------------------------
# Import reusable dataset implementation
# ------------------------------------------------------------

from nano_vlm.data.dataset import SyntheticVLDataset


# ------------------------------------------------------------
# Artifact directories
# ------------------------------------------------------------

DATASET_DIR = PROJECT_ROOT / "artifacts" / "dataset"
MODEL_ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "models"
FIGURES_DIR = PROJECT_ROOT / "assets" / "figures"

MODEL_ARTIFACTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

FIGURES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# ------------------------------------------------------------
# Load dataset artifacts created in Notebook 01
# ------------------------------------------------------------

train_samples = torch.load(
    DATASET_DIR / "train_samples.pt",
    weights_only=False,
)

val_samples = torch.load(
    DATASET_DIR / "val_samples.pt",
    weights_only=False,
)


# ------------------------------------------------------------
# Recreate datasets
# ------------------------------------------------------------

train_dataset = SyntheticVLDataset(train_samples)
val_dataset = SyntheticVLDataset(val_samples)


# ------------------------------------------------------------
# Recreate DataLoaders
# ------------------------------------------------------------

BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)


# ------------------------------------------------------------
# Verify inputs
# ------------------------------------------------------------

batch = next(iter(train_loader))

print("Device:", DEVICE)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

print("\nBatch keys:")
print(batch.keys())

print("\nImage shape:")
print(batch["image"].shape)

print("\nExample caption:")
print(batch["caption"][0])

Device: cpu
Train samples: 194
Validation samples: 49

Batch keys:
dict_keys(['image', 'caption', 'color', 'shape', 'position'])

Image shape:
torch.Size([32, 3, 32, 32])

Example caption:
green triangle bottom-right


# 3. Text Configuration

The text encoder needs a few architectural values:

- `VOCAB_SIZE`: number of tokens in the vocabulary
- `EMBED_DIM`: transformer hidden dimension
- `ATTENTION_HEADS`: number of attention heads
- `CONTEXT_WINDOW`: maximum sequence length
- `PROJECTION_DIM`: final shared embedding dimension

If these variables already exist in the notebook, we reuse them.

The most important compatibility requirement is:

$$
D \bmod H = 0
$$

where:

- $D$ = embedding dimension
- $H$ = number of attention heads

This is required because PyTorch divides the embedding dimension across the attention heads.

For example:

$$
64 / 4 = 16
$$

so 64 dimensions can be divided evenly across 4 attention heads.

In [54]:
# Reuse existing configuration when available.
# These fallback values are only used if the notebook has not
# defined the corresponding variables yet.

if "EMBED_DIM" not in globals():
    EMBED_DIM = 64

if "ATTENTION_HEADS" not in globals():
    ATTENTION_HEADS = 4

if "CONTEXT_WINDOW" not in globals():
    CONTEXT_WINDOW = 16

if "PROJECTION_DIM" not in globals():
    PROJECTION_DIM = 64

if "DEVICE" not in globals():
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


print("EMBED_DIM:", EMBED_DIM)
print("ATTENTION_HEADS:", ATTENTION_HEADS)
print("CONTEXT_WINDOW:", CONTEXT_WINDOW)
print("PROJECTION_DIM:", PROJECTION_DIM)
print("DEVICE:", DEVICE)

assert EMBED_DIM % ATTENTION_HEADS == 0, (
    "EMBED_DIM must be divisible by ATTENTION_HEADS."
)

EMBED_DIM: 64
ATTENTION_HEADS: 4
CONTEXT_WINDOW: 16
PROJECTION_DIM: 64
DEVICE: cpu


# 4. Tokenization Bridge

The dataset notebook provides captions as strings.

For example:

```text
"red circle top-left"

It needs integer token IDs:
[7, 12, 19, ...]

Each integer is used to look up a vector in an embedding table.

Our dataset is extremely small and controlled, so we do not need a complicated tokenizer.

We can use a simple whitespace tokenizer with special tokens:

<PAD>
<UNK>
<BOS>
<EOS>

This is still a from-scratch tokenizer and does not introduce a pretrained language model or external tokenizer.

In [55]:
# Create a small tokenizer only if one does not already exist.

if "VOCAB_SIZE" not in globals():

    SPECIAL_TOKENS = ["<PAD>", "<UNK>", "<BOS>", "<EOS>"]

    # The synthetic dataset captions are built from these words.
    DATASET_WORDS = [
        "red",
        "green",
        "blue",
        "yellow",
        "purple",
        "orange",
        "pink",
        "brown",
        "gray",
        "square",
        "circle",
        "triangle",
        "top-left",
        "top-center",
        "top-right",
        "middle-left",
        "center",
        "middle-right",
        "bottom-left",
        "bottom-center",
        "bottom-right",
    ]

    VOCAB = SPECIAL_TOKENS + DATASET_WORDS

    stoi = {
        token: index
        for index, token in enumerate(VOCAB)
    }

    itos = {
        index: token
        for token, index in stoi.items()
    }

    VOCAB_SIZE = len(VOCAB)

    PAD_TOKEN_ID = stoi["<PAD>"]
    UNK_TOKEN_ID = stoi["<UNK>"]
    BOS_TOKEN_ID = stoi["<BOS>"]
    EOS_TOKEN_ID = stoi["<EOS>"]

else:
    print("Using existing VOCAB_SIZE:", VOCAB_SIZE)

print("Vocabulary size:", VOCAB_SIZE)

Using existing VOCAB_SIZE: 25
Vocabulary size: 25


## Tokenizer

The tokenizer performs three simple operations:

1. Split the caption into words.
2. Convert each word into an integer ID.
3. Add special beginning/end tokens and pad to a fixed context window.

For example:

```text
"red circle top-left"

becomes conceptually:

<BOS> red circle top-left <EOS> <PAD> 

In [56]:
def tokenize_caption(caption, max_length=CONTEXT_WINDOW):
    """
    Convert one caption into a fixed-length tensor of token IDs.
    """

    words = caption.lower().split()

    token_ids = [BOS_TOKEN_ID]

    for word in words:
        token_ids.append(
            stoi.get(word, UNK_TOKEN_ID)
        )

    token_ids.append(EOS_TOKEN_ID)

    # Truncate if necessary.
    token_ids = token_ids[:max_length]

    # Pad to the fixed context window.
    token_ids += [
        PAD_TOKEN_ID
    ] * (max_length - len(token_ids))

    return torch.tensor(
        token_ids,
        dtype=torch.long,
    )

In [57]:
# Test the tokenizer on one caption

example_caption = "red circle top-left"

example_tokens = tokenize_caption(example_caption)

print("Caption:")
print(example_caption)

print("\nToken IDs:")
print(example_tokens)

print("\nToken tensor shape:")
print(example_tokens.shape)

assert example_tokens.shape == (CONTEXT_WINDOW,)

Caption:
red circle top-left

Token IDs:
tensor([ 2,  4, 14, 16,  3,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0])

Token tensor shape:
torch.Size([16])


# 5. Batch Tokenization

The tokenizer above processes one caption.

During training, we need to process a complete batch of captions.

The result should be:

$$
[B,L]
$$

For example, with a batch size of 32 and a context window of 16:

$$
[32,16]
$$

Each row corresponds to one caption.

The image and text samples remain aligned:

```text
Image 1  ↔ Caption 1
Image 2  ↔ Caption 2
Image 3  ↔ Caption 3
...


This alignment will become important when we calculate the diagonal positive pairs in the CLIP similarity matrix.

In [58]:
def tokenize_captions(captions, max_length=CONTEXT_WINDOW):
    """
    Tokenize a batch of caption strings.

    Returns:
        Tensor with shape [B, L]
    """

    return torch.stack(
        [
            tokenize_caption(
                caption,
                max_length=max_length,
            )
            for caption in captions
        ]
    )

# 6. Implement the Text Encoder

We can now implement the Transformer-based text encoder.

The main components are:

### Token Embedding

```text
[B, L]
   ↓
[B, L, D]

Each token ID is mapped to a learned vector.

Positional Embedding

Self-attention does not inherently know the order of tokens.

Therefore, we add a learned positional vector to every token representation.

Multi-Head Self-Attention

Each token can interact with the other tokens in the caption.

For example:

red
circle
top-left

can attend to one another.

This allows the representation of a token to depend on its surrounding context.

Layer Normalization

Layer normalization stabilizes the activations flowing through the Transformer block.

Sequence Representation

The Transformer produces one vector for every token:

[B,L,D]

We need one vector representing the entire caption.

For this small implementation, we use the representation at the <EOS> position.

Projection

Finally:

[B,D]→[B,P]

where $P=64$ for our shared embedding space.

L2 Normalization

This produces unit-length text embeddings suitable for later cosine-similarity-based contrastive learning.

In [59]:
class TextEncoder(nn.Module):
    """
    Small Transformer-based text encoder for the NanoVLM.

    Input:
        [B, L] token IDs

    Output:
        [B, projection_dim] normalized text embeddings
    """

    def __init__(
        self,
        vocab_size,
        embed_dim,
        attention_heads,
        max_length,
        projection_dim,
        padding_idx=0,
    ):
        super().__init__()

        # Convert token IDs into learned dense vectors.
        self.token_embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=padding_idx,
        )

        # Learned positional embeddings.
        self.position_embedding = nn.Embedding(
            num_embeddings=max_length,
            embedding_dim=embed_dim,
        )

        # Multi-head self-attention.
        self.self_attention = nn.MultiheadAttention(
            embed_dim=embed_dim,
            num_heads=attention_heads,
            batch_first=True,
        )

        # Layer normalization after the attention block.
        self.layer_norm = nn.LayerNorm(embed_dim)

        # Projection into the shared image/text embedding space.
        self.projection = nn.Linear(
            embed_dim,
            projection_dim,
        )

        self.max_length = max_length

    def forward(self, tokens):
        """
        Args:
            tokens:
                [B, L]

        Returns:
            normalized text embeddings:
                [B, projection_dim]
        """

        # --------------------------------------------------
        # 1. Token embeddings
        # [B, L] -> [B, L, D]
        # --------------------------------------------------

        x = self.token_embedding(tokens)

        batch_size, sequence_length, _ = x.shape

        if sequence_length > self.max_length:
            raise ValueError(
                f"Sequence length {sequence_length} exceeds "
                f"maximum length {self.max_length}."
            )

        # --------------------------------------------------
        # 2. Positional embeddings
        # [L] -> [L, D]
        # --------------------------------------------------

        positions = torch.arange(
            sequence_length,
            device=tokens.device,
        )

        position_embeddings = self.position_embedding(
            positions
        )

        # [B, L, D] + [L, D]
        x = x + position_embeddings

        # --------------------------------------------------
        # 3. Multi-head self-attention
        # --------------------------------------------------

        attention_output, _ = self.self_attention(
            query=x,
            key=x,
            value=x,
        )

        # Residual connection + LayerNorm
        x = self.layer_norm(
            x + attention_output
        )

        # --------------------------------------------------
        # 4. Select one representation for the sequence
        #
        # We use the <EOS> representation.
        # --------------------------------------------------

        eos_mask = tokens == EOS_TOKEN_ID

        # The tokenizer guarantees an EOS token unless
        # the sequence was truncated before reaching it.
        eos_positions = eos_mask.int().argmax(dim=1)

        batch_indices = torch.arange(
            batch_size,
            device=tokens.device,
        )

        text_representation = x[
            batch_indices,
            eos_positions,
        ]

        # [B, D]

        # --------------------------------------------------
        # 5. Projection
        # [B, D] -> [B, P]
        # --------------------------------------------------

        text_embedding = self.projection(
            text_representation
        )

        # --------------------------------------------------
        # 6. L2 normalization
        # [B, P] -> [B, P]
        # --------------------------------------------------

        text_embedding = F.normalize(
            text_embedding,
            dim=-1,
        )

        return text_embedding

# 7. Why Use the `<EOS>` Representation?

The Transformer produces a representation for every token:

$$
[B,L,D]
$$

But CLIP-style alignment requires one vector representing the complete caption.

We therefore select one token representation.

Here we use the `<EOS>` token.

Because the self-attention layer allows tokens to interact with one another, the `<EOS>` representation can contain information gathered from the entire sequence.

Conceptually:

```text
<BOS> red circle top-left <EOS>
  │     │      │      │      │
  └─────┴──────┴──────┴──────┘
             attention
                 ↓
          EOS representation
                 ↓
              [B, D]

# 8. Instantiate the Text Encoder

We now create the model using the configuration already defined above.

The final projection dimension must match the image encoder's embedding dimension.

For our project:

$$
P = 64
$$

Therefore:

```text
Image Encoder → [B, 64]

Text Encoder  → [B, 64]

In [60]:
text_encoder = TextEncoder(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    attention_heads=ATTENTION_HEADS,
    max_length=CONTEXT_WINDOW,
    projection_dim=PROJECTION_DIM,
    padding_idx=PAD_TOKEN_ID,
).to(DEVICE)

print(text_encoder)

TextEncoder(
  (token_embedding): Embedding(25, 64, padding_idx=0)
  (position_embedding): Embedding(16, 64)
  (self_attention): MultiheadAttention(
    (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
  )
  (layer_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True, bias=True)
  (projection): Linear(in_features=64, out_features=64, bias=True)
)


In [61]:
# Verify that the text and image embedding dimensions match

print("Text projection dimension :", PROJECTION_DIM)

if "model" in globals():
    print(
        "Image embedding dimension:",
        model.projection.out_features,
    )

    assert PROJECTION_DIM == model.projection.out_features

print("✓ Shared embedding dimension is compatible.")

Text projection dimension : 64
Image embedding dimension: 64
✓ Shared embedding dimension is compatible.


# 9. Test the Text Encoder

Before using the real dataset, we first test the text encoder with tokenized captions.

The expected input is:

$$
[B,L]
$$

and the expected output is:

$$
[B,P]
$$

For our NanoVLM:

$$
P=64
$$

The test verifies that:

- token IDs are accepted,
- positional embeddings work,
- self-attention works,
- the sequence representation is extracted correctly,
- the projection works,
- normalization works,
- gradients can still flow through the model.

In [62]:
# Create a small test batch of captions

test_captions = [
    "red circle top-left",
    "blue square center",
    "green triangle bottom-right",
    "yellow circle top-right",
]

test_tokens = tokenize_captions(test_captions)

print("Token shape:", test_tokens.shape)
print("Token dtype:", test_tokens.dtype)

Token shape: torch.Size([4, 16])
Token dtype: torch.int64


In [63]:
# Move tokens to the same device as the text encoder

test_tokens = test_tokens.to(DEVICE)

text_encoder.eval()

with torch.no_grad():
    test_embeddings = text_encoder(test_tokens)

print("Input token shape:      ", test_tokens.shape)
print("Output embedding shape:", test_embeddings.shape)
print("Embedding dtype:        ", test_embeddings.dtype)
print("Embedding device:       ", test_embeddings.device)

assert test_embeddings.ndim == 2
assert test_embeddings.shape[0] == len(test_captions)
assert test_embeddings.shape[1] == PROJECTION_DIM

print("✓ Text encoder output shape is correct.")

Input token shape:       torch.Size([4, 16])
Output embedding shape: torch.Size([4, 64])
Embedding dtype:         torch.float32
Embedding device:        cpu
✓ Text encoder output shape is correct.


# 10. Shape Walkthrough

Let's explicitly trace the tensor through the text encoder.

Let:

- $B$ = batch size
- $L$ = sequence length
- $D$ = Transformer embedding dimension
- $P$ = shared projection dimension

The complete transformation is:

```text
Token IDs
[B, L]

      ↓

Token Embedding
[B, L, D]

      ↓

Positional Embedding
[B, L, D]

      ↓

Multi-Head Self-Attention
[B, L, D]

      ↓

Layer Normalization
[B, L, D]

      ↓

Select <EOS>
[B, D]

      ↓

Linear Projection
[B, P]

      ↓

L2 Normalization
[B, P]

# 11. Inspect Intermediate Shapes

To make the Transformer easier to understand, we can manually inspect the major tensor transformations.

This is purely an educational debugging step.

It lets us see exactly where the sequence dimension is preserved and where it is eventually reduced to one vector per caption.

In [64]:
# Inspect intermediate shapes

text_encoder.eval()

tokens = test_tokens

with torch.no_grad():

    # Token embeddings
    x = text_encoder.token_embedding(tokens)

    print("Token IDs:              ", tokens.shape)
    print("Token embeddings:       ", x.shape)

    # Positional embeddings
    sequence_length = tokens.shape[1]

    positions = torch.arange(
        sequence_length,
        device=tokens.device,
    )

    position_embeddings = text_encoder.position_embedding(
        positions
    )

    x = x + position_embeddings

    print("After positional add:   ", x.shape)

    # Self-attention
    attention_output, _ = text_encoder.self_attention(
        x,
        x,
        x,
    )

    print("Attention output:       ", attention_output.shape)

    # Residual + LayerNorm
    x = text_encoder.layer_norm(
        x + attention_output
    )

    print("After LayerNorm:        ", x.shape)

    # Select EOS representation
    eos_mask = tokens == EOS_TOKEN_ID
    eos_positions = eos_mask.int().argmax(dim=1)

    batch_indices = torch.arange(
        tokens.shape[0],
        device=tokens.device,
    )

    text_representation = x[
        batch_indices,
        eos_positions,
    ]

    print("EOS representation:     ", text_representation.shape)

    # Projection
    projected = text_encoder.projection(
        text_representation
    )

    print("Projected embedding:    ", projected.shape)

    # Normalization
    normalized = F.normalize(
        projected,
        dim=-1,
    )

    print("Normalized embedding:   ", normalized.shape)

Token IDs:               torch.Size([4, 16])
Token embeddings:        torch.Size([4, 16, 64])
After positional add:    torch.Size([4, 16, 64])
Attention output:        torch.Size([4, 16, 64])
After LayerNorm:         torch.Size([4, 16, 64])
EOS representation:      torch.Size([4, 64])
Projected embedding:     torch.Size([4, 64])
Normalized embedding:    torch.Size([4, 64])


# 12. Why Positional Embeddings Matter

Self-attention can compare every token with every other token, but the attention mechanism itself does not automatically know the order in which the tokens appeared.

Consider:

```text
red circle top-left

and:

top-left circle red

and:

top-left circle red

They contain similar words but have different ordering.

Our learnable positional embeddings provide the model with information about token position.

For each position:

0,1,2,…,L−1

we learn a vector and add it to the corresponding token embedding.
Therefore the Transformer receives both:

what the token is
where the token occurs

# 13. Why Multi-Head Self-Attention?

The self-attention layer allows each token to interact with the other tokens in the caption.

For example:

```text
red     circle     top-left
 │         │           │
 └─────────┼───────────┘
           │
      Self-Attention
           ↓
 Context-aware
 representations

# 14. L2 Normalization

The final text embedding is normalized using:

$$
\hat z_T =
\frac{z_T}{\|z_T\|_2}
$$

In [65]:
# Verify that normalized embeddings have approximately unit norm

embedding_norms = torch.norm(
    test_embeddings,
    dim=-1,
)

print("Embedding norms:")
print(embedding_norms)

assert torch.allclose(
    embedding_norms,
    torch.ones_like(embedding_norms),
    atol=1e-5,
)

print("✓ Text embeddings have approximately unit L2 norm.")

Embedding norms:
tensor([1.0000, 1.0000, 1.0000, 1.0000])
✓ Text embeddings have approximately unit L2 norm.


# 15. Test the Text Encoder on a Real Dataset Batch

Now we connect the text encoder to the existing dataset.

We do **not** recreate the dataset or DataLoader here.

The dataset notebook already provides the image-caption pairs.

Because the DataLoader may return a dictionary containing metadata, we first extract the captions without assuming that the batch contains exactly two values.

The expected flow is:

```text
Existing train_loader
        ↓
      batch
        ↓
    captions
        ↓
     tokenizer
        ↓
   token IDs [B, L]
        ↓
   Text Encoder
        ↓
text embeddings [B, 64]

In [66]:
# Get one real batch from the existing DataLoader.
# We do not recreate the dataset here.

batch = next(iter(train_loader))

print("Batch type:", type(batch))

if isinstance(batch, dict):
    captions = batch["caption"]
    images = batch["image"]

elif isinstance(batch, (tuple, list)):
    # This branch supports a Dataset that returns
    # multiple fields rather than a dictionary.
    #
    # Adjust these indices only if your existing Dataset
    # uses a different ordering.
    images = batch[0]
    captions = batch[1]

else:
    raise TypeError(
        f"Unsupported batch type: {type(batch)}"
    )

print("Images shape:", images.shape)
print("Number of captions:", len(captions))
print("First caption:", captions[0])

Batch type: <class 'dict'>
Images shape: torch.Size([32, 3, 32, 32])
Number of captions: 32
First caption: brown circle bottom-right


## Tokenize the Real Captions

The captions from the DataLoader are Python strings.

We convert the complete batch into a token tensor:

$$
[B] \rightarrow [B,L]
$$

The batch size is inferred automatically from the DataLoader.

We do not hard-code it.

In [67]:
# Convert real captions into token IDs

tokens = tokenize_captions(
    captions,
    max_length=CONTEXT_WINDOW,
)

tokens = tokens.to(DEVICE)

print("Token shape:", tokens.shape)
print("Token dtype:", tokens.dtype)
print("Token device:", tokens.device)

assert tokens.ndim == 2
assert tokens.shape[0] == images.shape[0]
assert tokens.shape[1] == CONTEXT_WINDOW
assert tokens.dtype == torch.long

Token shape: torch.Size([32, 16])
Token dtype: torch.int64
Token device: cpu


In [68]:
# Generate text embeddings from the real captions

text_encoder.eval()

with torch.no_grad():
    text_embeddings = text_encoder(tokens)

print("Tokens shape:           ", tokens.shape)
print("Text embeddings shape: ", text_embeddings.shape)
print("Text embeddings dtype: ", text_embeddings.dtype)
print("Text embeddings device:", text_embeddings.device)

assert text_embeddings.ndim == 2
assert text_embeddings.shape[0] == images.shape[0]
assert text_embeddings.shape[1] == PROJECTION_DIM

print("✓ Real captions successfully passed through the Text Encoder.")

Tokens shape:            torch.Size([32, 16])
Text embeddings shape:  torch.Size([32, 64])
Text embeddings dtype:  torch.float32
Text embeddings device: cpu
✓ Real captions successfully passed through the Text Encoder.


# 16. Verify Image/Text Dimension Compatibility

We now have both sides of the multimodal pipeline.

The image encoder produces:

$$
Z_I \in \mathbb{R}^{B \times 64}
$$

The text encoder produces:

$$
Z_T \in \mathbb{R}^{B \times 64}
$$

Therefore they can be compared directly.

```text
Images
[B, 3, 32, 32]
       ↓
Image Encoder
       ↓
[B, 64]


Captions
[B, L]
       ↓
Text Encoder
       ↓
[B, 64]

In [69]:
# Verify that the image and text embeddings have matching shapes

if "model" in globals():

    model.eval()

    with torch.no_grad():
        image_embeddings = model(
            images.to(next(model.parameters()).device)
        )

    print("Image embeddings:", image_embeddings.shape)
    print("Text embeddings :", text_embeddings.shape)

    assert image_embeddings.shape == text_embeddings.shape
    assert image_embeddings.shape[1] == PROJECTION_DIM

    print("✓ Image and text embeddings occupy the same dimensional space.")
else:
    print(
        "ImageEncoder model is not currently defined in this kernel."
    )
    print(
        "Text encoder output dimension:",
        text_embeddings.shape[1],
    )

Image embeddings: torch.Size([32, 64])
Text embeddings : torch.Size([32, 64])
✓ Image and text embeddings occupy the same dimensional space.


# 17. Sanity Checks

Before moving to contrastive learning, we want to verify a few basic properties.

The text encoder should:

1. Preserve the batch dimension.
2. Produce a two-dimensional output.
3. Produce the configured shared embedding dimension.
4. Produce normalized embeddings.
5. Accept token IDs on the correct device.
6. Remain fully differentiable.

These checks do not prove that the encoder has learned anything meaningful.

They only verify that the architecture and tensor pipeline are functioning correctly.

Learning will happen later when the encoder is trained using matching and mismatched image-text pairs.

In [70]:
# Basic output sanity checks

assert text_embeddings.ndim == 2

assert text_embeddings.shape[0] == tokens.shape[0]

assert text_embeddings.shape[1] == PROJECTION_DIM

norms = torch.norm(
    text_embeddings,
    dim=-1,
)

assert torch.allclose(
    norms,
    torch.ones_like(norms),
    atol=1e-5,
)

print("✓ Output is 2-dimensional.")
print("✓ Batch dimension is preserved.")
print("✓ Projection dimension is correct.")
print("✓ Embeddings are L2-normalized.")

✓ Output is 2-dimensional.
✓ Batch dimension is preserved.
✓ Projection dimension is correct.
✓ Embeddings are L2-normalized.


# 18. Verify Gradient Flow

The text encoder must be trainable.

The token embeddings, positional embeddings, attention parameters, normalization parameters, and projection layer should all be connected to the computation graph.

We can verify this by running a small forward/backward pass.

This does **not** train the model meaningfully yet.

It simply confirms that gradients can flow from the final text embedding back through the encoder.

In [71]:
# Verify gradient flow through the complete text encoder

text_encoder.train()

tokens_for_gradient = tokens.clone()

gradient_embeddings = text_encoder(
    tokens_for_gradient
)

# A simple scalar objective for checking backpropagation
dummy_loss = gradient_embeddings.sum()

dummy_loss.backward()

gradient_parameters = [
    parameter
    for parameter in text_encoder.parameters()
    if parameter.requires_grad
]

parameters_with_gradients = [
    parameter
    for parameter in gradient_parameters
    if parameter.grad is not None
]

print(
    "Trainable parameters:",
    len(gradient_parameters),
)

print(
    "Parameters with gradients:",
    len(parameters_with_gradients),
)

assert len(parameters_with_gradients) == len(
    gradient_parameters
)

print("✓ Gradients can flow through the Text Encoder.")

Trainable parameters: 10
Parameters with gradients: 10
✓ Gradients can flow through the Text Encoder.


In [72]:
# Clear the temporary gradients.
# The actual optimizer/training loop will be introduced later.

text_encoder.zero_grad(set_to_none=True)

print("Temporary gradient test complete.")

Temporary gradient test complete.


In [73]:
print("CONTEXT_WINDOW:", CONTEXT_WINDOW)
print("PROJECTION_DIM:", PROJECTION_DIM)

print("tokenize_caption exists:", "tokenize_caption" in globals())
print("tokenize_captions exists:", "tokenize_captions" in globals())

CONTEXT_WINDOW: 16
PROJECTION_DIM: 64
tokenize_caption exists: True
tokenize_captions exists: True


In [74]:
# ============================================================
# Save Text Encoder artifacts and outputs
# ============================================================

MODEL_ARTIFACTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 1. Save initial Text Encoder weights
# ------------------------------------------------------------

text_encoder_path = (
    MODEL_ARTIFACTS_DIR / "text_encoder_initial.pt"
)

torch.save(
    model.state_dict(),
    text_encoder_path,
)


# ------------------------------------------------------------
# 2. Get a real batch of captions
# ------------------------------------------------------------

model.eval()

with torch.no_grad():

    batch = next(iter(train_loader))

    captions = batch["caption"]

    # Use the project's existing tokenizer function
    tokens = tokenize_captions(
        captions,
        max_length=CONTEXT_WINDOW,
    )

    tokens = tokens.to(DEVICE)

    # Generate text embeddings
    text_embeddings = text_encoder(tokens)


# ------------------------------------------------------------
# 3. Verify expected shapes
# ------------------------------------------------------------

print("Number of captions:", len(captions))
print("Tokens shape:", tokens.shape)
print("Text embeddings shape:", text_embeddings.shape)

assert tokens.ndim == 2

assert tokens.shape[0] == len(captions)

assert text_embeddings.ndim == 2

assert text_embeddings.shape[0] == len(captions)

assert text_embeddings.shape[1] == PROJECTION_DIM


# ------------------------------------------------------------
# 4. Save representative token batch
# ------------------------------------------------------------

tokens_path = (
    MODEL_ARTIFACTS_DIR / "text_tokens_initial.pt"
)

torch.save(
    tokens.cpu(),
    tokens_path,
)


# ------------------------------------------------------------
# 5. Save representative text embeddings
# ------------------------------------------------------------

text_embeddings_path = (
    MODEL_ARTIFACTS_DIR / "text_embeddings_initial.pt"
)

torch.save(
    text_embeddings.cpu(),
    text_embeddings_path,
)


# ------------------------------------------------------------
# 6. Save tokenizer configuration
# ------------------------------------------------------------

tokenizer_config = {
    "context_window": CONTEXT_WINDOW,
}

# Save vocabulary if it exists in the notebook
if "VOCAB" in globals():
    tokenizer_config["vocab"] = VOCAB

if "vocab" in globals():
    tokenizer_config["vocab"] = vocab

if "TOKEN_TO_ID" in globals():
    tokenizer_config["token_to_id"] = TOKEN_TO_ID

if "token_to_id" in globals():
    tokenizer_config["token_to_id"] = token_to_id

tokenizer_config_path = (
    MODEL_ARTIFACTS_DIR / "tokenizer_config.pt"
)

torch.save(
    tokenizer_config,
    tokenizer_config_path,
)


# ------------------------------------------------------------
# 7. Confirmation
# ------------------------------------------------------------

print("\nText Encoder artifacts saved:")

print("✓", text_encoder_path)
print("✓", tokens_path)
print("✓", text_embeddings_path)
print("✓", tokenizer_config_path)

Number of captions: 32
Tokens shape: torch.Size([32, 16])
Text embeddings shape: torch.Size([32, 64])

Text Encoder artifacts saved:
✓ d:\github\Build A NanoVLM From SCRATCH\artifacts\models\text_encoder_initial.pt
✓ d:\github\Build A NanoVLM From SCRATCH\artifacts\models\text_tokens_initial.pt
✓ d:\github\Build A NanoVLM From SCRATCH\artifacts\models\text_embeddings_initial.pt
✓ d:\github\Build A NanoVLM From SCRATCH\artifacts\models\tokenizer_config.pt


# 19. Text Encoder Parameter Count

As with the image encoder, it is useful to know how many trainable parameters the text encoder contains.

We can calculate the count directly rather than assuming it.

A compact parameter count is desirable for our NanoVLM because the project is intentionally designed to be:

- small
- understandable
- trainable on modest hardware
- easy to inspect

The goal is not to reproduce the size of a production language model.

The goal is to expose the core mechanism behind multimodal representation learning.

In [75]:
# Count trainable Text Encoder parameters

text_encoder_parameters = sum(
    parameter.numel()
    for parameter in text_encoder.parameters()
    if parameter.requires_grad
)

print(
    f"Text Encoder trainable parameters: "
    f"{text_encoder_parameters:,}"
)

Text Encoder trainable parameters: 23,552


# 20. Final Text Encoder Architecture

The complete text pipeline is now:

```text
Caption
"red circle top-left"
        ↓
Tokenizer
        ↓
Token IDs
[B, L]
        ↓
Token Embedding
[B, L, D]
        ↓
Positional Embedding
[B, L, D]
        ↓
Multi-Head Self-Attention
[B, L, D]
        ↓
Residual + LayerNorm
[B, L, D]
        ↓
<EOS> Representation
[B, D]
        ↓
Linear Projection
[B, 64]
        ↓
L2 Normalization
[B, 64]

# 21. Text Encoder Complete

The Text Encoder is now implemented completely from scratch.

We have implemented:

- Token embeddings
- Learnable positional embeddings
- Multi-head self-attention
- Layer normalization
- Sequence-level `<EOS>` representation
- Linear projection
- L2 normalization
- Real-batch testing
- Tensor shape verification
- Parameter counting
- Gradient-flow verification

The final interface is:

$$
[B,L]
\rightarrow
[B,64]
$$

while the Image Encoder provides:

$$
[B,3,32,32]
\rightarrow
[B,64]
$$

Both modalities therefore produce representations in:

$$
\mathbb{R}^{64}
$$

The next stage will combine the Image Encoder and Text Encoder and implement the **shared embedding space + CLIP contrastive loss**.

```text
Image
  ↓
Image Encoder
  ↓
Image Embedding
  ↓
Normalize
  │
  ├──── Similarity Matrix ────┐
  │                           │
Text Embedding                │
  ↑                           │
Text Encoder                  │
  ↑                           │
Tokens                        │
                              ↓
                       Contrastive Loss